# 🪐 Recreating Planetary Topography (Earth, Mars, Moon)

This notebook demonstrates how to recreate the three main planetary topography globes described in Section 3.1 of Koelemeijer & Winterbourne (2021). 

### 🌎 Scientific Context
The topography of a planet is a fingerprint of its interior dynamics:
- **Earth**: Active plate tectonics constantly rework the surface, creating distinct ocean ridges, subduction trenches, and mountain belts (vertical exaggeration: **50:1**).
- **Mars**: A transitional body showing massive volcanoes (Olympus Mons) and a canyon system (Valles Marineris) alongside impact craters (vertical exaggeration: **20:1**).
- **The Moon**: An inactive body covered in impact craters from external collisions (vertical exaggeration: **14:1**).

We can print them at the **same size** (e.g., 80 mm diameter) to study surface details, or at the **correct relative scale** (Earth: 40 mm radius, Mars: 21.3 mm, Moon: 10.9 mm).

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    calculate_displacement_scale
)

## Step 2: Recreate the Earth Globe (50:1 Exaggeration)

We use the preloaded `ETOPO` topography grid.

In [ ]:
earth_radius_mm = 40.0
earth_exagg = 50.0

# 1. Initialize model
earth_model = GlobeModel(n_points=5000, radius=earth_radius_mm)

# 2. Load topography dataset
topo_grid = GeographicGrid.from_netcdf(
    "../../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
grid_ds = GeographicGrid(
    lats=topo_grid.lats[::15],  # Downsample for speed in demo
    lons=topo_grid.lons[::15],
    grid=topo_grid.grid[::15, ::15]
)

# 3. Calculate scale and apply displacement
scale = calculate_displacement_scale(earth_radius_mm, vertical_exagg=earth_exagg, grid_units='m')
earth_model.outer.displace(GridDisplacer(grid_ds), scale=scale)

# 4. Export mesh
os.makedirs('../../outputs', exist_ok=True)
earth_model.export("../../outputs/paper_earth_topography.stl")
print("Earth topography model exported!")

## Step 3: Recreate Mars & Moon Globes (Placeholder Data)

To print the real Mars (MOLA) and Moon (LOLA) grids, download the NetCDF grids from the [Data Sourcing Guide](../../user_guide/8_where_to_get_data.md) and swap the file paths in the code below.

Here, we simulate Mars/Moon topography using synthetic cratered grids to demonstrate the code.

In [ ]:
# Create synthetic cratered grid (placeholder)
lats = np.linspace(-90, 90, 180)
lons = np.linspace(-180, 180, 360)
LON, LAT = np.meshgrid(lons, lats)
# Add simulated impact basins/craters
z = -2000.0 * np.sin(np.radians(LAT)) + 1500.0 * np.cos(3 * np.radians(LON)) * np.cos(3 * np.radians(LAT))
placeholder_grid = GeographicGrid(lats=lats, lons=lons, grid=z)

# --- MARS MODEL (20:1 Exaggeration) ---
# Mars is printed at 21.3 mm radius to be in scale with Earth
mars_radius_mm = 21.3
mars_exagg = 20.0
mars_model = GlobeModel(n_points=5000, radius=mars_radius_mm)
mars_scale = calculate_displacement_scale(mars_radius_mm, vertical_exagg=mars_exagg, grid_units='m')
mars_model.outer.displace(GridDisplacer(placeholder_grid), scale=mars_scale)
mars_model.export("../../outputs/paper_mars_topography.stl")

# --- MOON MODEL (14:1 Exaggeration) ---
# Moon is printed at 10.9 mm radius to be in scale with Earth
moon_radius_mm = 10.9
moon_exagg = 14.0
moon_model = GlobeModel(n_points=3000, radius=moon_radius_mm)
moon_scale = calculate_displacement_scale(moon_radius_mm, vertical_exagg=moon_exagg, grid_units='m')
moon_model.outer.displace(GridDisplacer(placeholder_grid), scale=moon_scale)
moon_model.export("../../outputs/paper_moon_topography.stl")

print("Mars and Moon model files generated using placeholder grids!")

## Step 4: Preview the Three Globes in 3D

In [ ]:
fig = plt.figure(figsize=(15, 5))

# Earth
ax1 = fig.add_subplot(131, projection='3d')
pts_e = earth_model.outer.vertices
ax1.scatter(pts_e[::5, 0], pts_e[::5, 1], pts_e[::5, 2], c=pts_e[::5, 2], cmap='gist_earth', s=1)
ax1.set_title(f"Earth (Radius: {earth_radius_mm} mm)")

# Mars
ax2 = fig.add_subplot(132, projection='3d')
pts_ma = mars_model.outer.vertices
ax2.scatter(pts_ma[::5, 0], pts_ma[::5, 1], pts_ma[::5, 2], c='chocolate', s=1)
ax2.set_title(f"Mars (Radius: {mars_radius_mm} mm)")

# Moon
ax3 = fig.add_subplot(133, projection='3d')
pts_mo = moon_model.outer.vertices
ax3.scatter(pts_mo[::3, 0], pts_mo[::3, 1], pts_mo[::3, 2], c='darkgray', s=1)
ax3.set_title(f"Moon (Radius: {moon_radius_mm} mm)")

plt.show()